# AI vs Real Detector — CNNDetection + FFHQ + LSUN Dataset Prep

Run this **only on Google Colab**. Archives are downloaded to Colab/Drive; nothing is downloaded to this repository or your laptop.

Sources:

- **CNNDetection** train / val / test ([Wang et al., CVPR 2020](https://github.com/PeterWang512/CNNDetection/tree/master/dataset))
  - Train and val **real** images are **LSUN** (not LSNU)
  - Train and val **fake** images are **ProGAN**
  - Test covers 13 generators (ProGAN, StyleGAN, StyleGAN2, BigGAN, CycleGAN, StarGAN, GauGAN, CRN, IMLE, SITD, SAN, Deepfake, WhichFaceIsReal)
- Extra **FFHQ** real faces mixed into train / val / test real folders

Output layout expected by `src/train.py`:

```text
data/
  train/{real,fake}/<source>/*.jpg
  val/{real,fake}/<source>/*.jpg
  test/{real,fake}/<source>/*.jpg
```

The default uses every image in the proper three-way protocol: the official **train**, **val**, and 13-generator **test** archives. The training archive is ~70 GB and needs substantial temporary Colab/Drive space. A small-space mode exists, but it re-splits the official validation archive and must not be described as an official train/val experiment.


In [ ]:
# 1. Mount Drive first, then configure paths (archives cache on Drive)
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, random, shutil, json, subprocess, hashlib

SEED = 42
random.seed(SEED)

DRIVE_ROOT = Path("/content/drive/MyDrive/ai-vs-real-face-detector")
DATA_DIR = DRIVE_ROOT / "data"
CACHE = DRIVE_ROOT / "cnn_detection_cache"
LOCAL_EXTRACT = Path("/content/cnn_detection_extract")

HF_REPO = "sywang/CNNDetection"

# Proper experiment: separate official LSUN/ProGAN train and validation archives.
# This needs ~70 GB of downloads plus extraction workspace; use a high-storage Colab runtime.
DOWNLOAD_FULL_TRAINSET = True
# Set True only if storage prevents the official train archive. This creates a
# non-official train/val split from progan_val.zip and is for smoke tests only.
ALLOW_VAL_AS_TRAIN_FALLBACK = False
# Official 13-generator test zip ~20GB. False = ProGAN-only test zip (~0.8GB).
DOWNLOAD_FULL_TESTSET = True

# None uses every image. Set finite limits only for a quick Colab pilot.
MAX_TRAIN_PER_CATEGORY = None
MAX_VAL_PER_CATEGORY = None
MAX_TEST_PER_LEAF = None
N_FFHQ_REAL = 2500

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

for split in ["train", "val", "test"]:
    for label in ["real", "fake"]:
        (DATA_DIR / split / label).mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
LOCAL_EXTRACT.mkdir(parents=True, exist_ok=True)

assert DOWNLOAD_FULL_TRAINSET or ALLOW_VAL_AS_TRAIN_FALLBACK, (
    "Use the official training split, or explicitly opt into the non-official fallback."
)
print("DATA_DIR:", DATA_DIR)
print("CACHE (Drive):", CACHE)
print("Temporary extraction (Colab VM):", LOCAL_EXTRACT)


In [ ]:
# 2. Install unpack tools
!apt-get -qq install -y p7zip-full
!pip -q install -U kaggle huggingface_hub


In [ ]:
# 3. Helpers
from huggingface_hub import hf_hub_download

def image_files(root):
    root = Path(root)
    if not root.exists():
        return []
    return sorted([
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ])

def hf_file(filename):
    path = hf_hub_download(
        repo_id=HF_REPO,
        repo_type="dataset",
        filename=filename,
        cache_dir=str(CACHE / "hf"),
    )
    print("Ready:", filename, Path(path).stat().st_size)
    return Path(path)

def unzip_to(archive, dest):
    dest = Path(dest)
    marker = dest / "_done.txt"
    if marker.exists():
        print("Already extracted:", dest)
        return dest
    shutil.rmtree(dest, ignore_errors=True)
    dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["unzip", "-q", str(archive), "-d", str(dest)])
    marker.write_text("ok")
    return dest

def copy_capped(files, dest_dir, cap, seed, prefix):
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    rng = random.Random(seed)
    files = list(files)
    rng.shuffle(files)
    selected = files[:cap] if cap is not None else files
    for idx, src in enumerate(selected):
        src = Path(src)
        dst = dest_dir / f"{prefix}_{idx:06d}{src.suffix.lower()}"
        if not dst.exists():
            shutil.copy2(src, dst)
    return len(selected)

def source_seed(seed, source):
    # Python's hash() is deliberately randomized per process; use a stable
    # digest so the selected image subset is reproducible across Colab runs.
    digest = hashlib.sha256(str(source).encode("utf-8")).digest()
    return seed + int.from_bytes(digest[:4], "big")

def iter_leaf_label_dirs(root):
    root = Path(root)
    if not root.exists():
        return
    for folder in [root, *root.rglob("*")]:
        if not folder.is_dir():
            continue
        name = folder.name.lower()
        if name in {"0_real", "real"}:
            yield folder, "real"
        elif name in {"1_fake", "fake"}:
            yield folder, "fake"


In [ ]:
# 4. Download CNNDetection validation set (LSUN real + ProGAN fake)
VAL_ZIP = hf_file("progan_val.zip")
VAL_EXTRACT = unzip_to(VAL_ZIP, LOCAL_EXTRACT / "val")
print("Val images:", len(image_files(VAL_EXTRACT)))


In [ ]:
# 5. Download CNNDetection test set
if DOWNLOAD_FULL_TESTSET:
    TEST_ZIP = hf_file("CNN_synth_testset.zip")
else:
    TEST_ZIP = hf_file("progan_testset.zip")
TEST_EXTRACT = unzip_to(TEST_ZIP, LOCAL_EXTRACT / "test")
print("Test images:", len(image_files(TEST_EXTRACT)))


In [ ]:
# 6. Official CNNDetection training set (ProGAN + LSUN, 7z parts ~70GB)
TRAIN_EXTRACT = LOCAL_EXTRACT / "train"
if DOWNLOAD_FULL_TRAINSET:
    if not (TRAIN_EXTRACT / "_done.txt").exists():
        TRAIN_EXTRACT.mkdir(parents=True, exist_ok=True)
        parts = [f"progan_train.7z.{i:03d}" for i in range(1, 8)]
        local_parts = [hf_file(part) for part in parts]
        first = local_parts[0]
        subprocess.check_call(["7z", "x", str(first), f"-o{TRAIN_EXTRACT}", "-y"])
        inner_zip = list(TRAIN_EXTRACT.rglob("progan_train.zip"))
        if inner_zip:
            subprocess.check_call(["unzip", "-q", str(inner_zip[0]), "-d", str(TRAIN_EXTRACT)])
            inner_zip[0].unlink(missing_ok=True)
        (TRAIN_EXTRACT / "_done.txt").write_text("ok")
    print("Train images (full archive, before capping):", len(image_files(TRAIN_EXTRACT)))
else:
    assert ALLOW_VAL_AS_TRAIN_FALLBACK
    print("WARNING: using a validation-derived training split; this is not the official protocol.")


In [ ]:
# 7. Kaggle auth + FFHQ real faces
from google.colab import files

KAGGLE_PATH = Path("/root/.kaggle/kaggle.json")
KAGGLE_PATH.parent.mkdir(parents=True, exist_ok=True)
if not KAGGLE_PATH.exists():
    print("Upload kaggle.json")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No kaggle.json uploaded.")
    shutil.move(next(iter(uploaded)), KAGGLE_PATH)
os.chmod(KAGGLE_PATH, 0o600)

FFHQ_DL = Path("/content/ffhq_dl")
FFHQ_SOURCE = DRIVE_ROOT / "source" / "ffhq"
FFHQ_SOURCE.mkdir(parents=True, exist_ok=True)

existing_ffhq = image_files(FFHQ_SOURCE)
print("Existing FFHQ source:", len(existing_ffhq))

if len(existing_ffhq) < N_FFHQ_REAL:
    if not (FFHQ_DL / "download_complete.txt").exists():
        shutil.rmtree(FFHQ_DL, ignore_errors=True)
        FFHQ_DL.mkdir(parents=True, exist_ok=True)
        !kaggle datasets download -d pankymathur/ffhq-224k -p /content/ffhq_dl --unzip
        (FFHQ_DL / "download_complete.txt").touch()
    candidates = image_files(FFHQ_DL)
    print("FFHQ candidates:", len(candidates))
    assert len(candidates) >= N_FFHQ_REAL, f"Need {N_FFHQ_REAL} FFHQ images, found {len(candidates)}"
    random.seed(SEED)
    random.shuffle(candidates)
    for idx, src in enumerate(candidates[:N_FFHQ_REAL]):
        dst = FFHQ_SOURCE / f"ffhq_{idx:06d}{src.suffix.lower()}"
        if not dst.exists():
            shutil.copy2(src, dst)

existing_ffhq = image_files(FFHQ_SOURCE)
print("FFHQ source ready:", len(existing_ffhq))


In [ ]:
# 8. Rebuild Drive data/ from CNNDetection + FFHQ
for split in ["train", "val", "test"]:
    for label in ["real", "fake"]:
        dest = DATA_DIR / split / label
        if dest.exists():
            shutil.rmtree(dest)
        dest.mkdir(parents=True, exist_ok=True)

def ingest_split(extract_root, split, cap, seed_base):
    counts = {"real": 0, "fake": 0}
    for folder, label in iter_leaf_label_dirs(extract_root):
        rel = folder.relative_to(extract_root)
        source_parts = [p for p in rel.parts if p.lower() not in {"0_real", "1_fake", "real", "fake"}]
        source = "_".join(source_parts) if source_parts else ("lsun" if label == "real" else "progan")
        dest = DATA_DIR / split / label / source
        n = copy_capped(image_files(folder), dest, cap, source_seed(seed_base, source), f"{split}_{source}")
        counts[label] += n
        print(f"{split}/{label}/{source}: {n}")
    return counts

if DOWNLOAD_FULL_TRAINSET:
    train_counts = ingest_split(TRAIN_EXTRACT, "train", MAX_TRAIN_PER_CATEGORY, SEED)
    val_counts = ingest_split(VAL_EXTRACT, "val", MAX_VAL_PER_CATEGORY, SEED + 1)
else:
    # Hold out 20% of each official-val leaf as val; remainder becomes train.
    train_counts = {"real": 0, "fake": 0}
    val_counts = {"real": 0, "fake": 0}
    for folder, label in iter_leaf_label_dirs(VAL_EXTRACT):
        rel = folder.relative_to(VAL_EXTRACT)
        source_parts = [p for p in rel.parts if p.lower() not in {"0_real", "1_fake", "real", "fake"}]
        source = "_".join(source_parts) if source_parts else ("lsun" if label == "real" else "progan")
        files_ = image_files(folder)
        rng = random.Random(source_seed(SEED, source))
        files_ = files_.copy()
        rng.shuffle(files_)
        n_val = max(1, int(len(files_) * 0.20)) if len(files_) > 1 else 0
        val_files = files_[:n_val]
        train_files = files_[n_val:]
        train_cap = min(len(train_files), MAX_TRAIN_PER_CATEGORY)
        val_cap = min(len(val_files), MAX_VAL_PER_CATEGORY)
        n_tr = copy_capped(train_files, DATA_DIR / "train" / label / source, train_cap, source_seed(SEED, f"train/{source}"), f"train_{source}")
        n_va = copy_capped(val_files, DATA_DIR / "val" / label / source, val_cap, source_seed(SEED, f"val/{source}"), f"val_{source}")
        train_counts[label] += n_tr
        val_counts[label] += n_va
        print(f"train/{label}/{source}: {n_tr}  val/{label}/{source}: {n_va}")

test_counts = ingest_split(TEST_EXTRACT, "test", MAX_TEST_PER_LEAF, SEED + 2)
print("CNNDetection ingest:", {"train": train_counts, "val": val_counts, "test": test_counts})


In [ ]:
# 9. Mix FFHQ reals into train/val/test (70/15/15)
ffhq = image_files(FFHQ_SOURCE)
rng = random.Random(SEED)
rng.shuffle(ffhq)
n = min(len(ffhq), N_FFHQ_REAL)
ffhq = ffhq[:n]
n_train = int(n * 0.70)
n_val = int(n * 0.15)
splits = {
    "train": ffhq[:n_train],
    "val": ffhq[n_train:n_train + n_val],
    "test": ffhq[n_train + n_val:],
}
for split, files_ in splits.items():
    copied = copy_capped(files_, DATA_DIR / split / "real" / "ffhq", None, SEED, f"ffhq_{split}")
    print(f"{split}/real/ffhq: {copied}")


In [ ]:
# 10. Verify and write manifest
def count_images(path):
    return sum(1 for p in Path(path).rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)

counts = {}
for split in ["train", "val", "test"]:
    counts[split] = {
        "real": count_images(DATA_DIR / split / "real"),
        "fake": count_images(DATA_DIR / split / "fake"),
    }
print(json.dumps(counts, indent=2))

for split in ["train", "val", "test"]:
    assert counts[split]["real"] > 0, f"No real images in {split}"
    assert counts[split]["fake"] > 0, f"No fake images in {split}"

manifest = {
    "seed": SEED,
    "source": "CNNDetection (LSUN real + ProGAN; 13-generator test) + FFHQ reals",
    "protocol": (
        "official train/val/test" if DOWNLOAD_FULL_TRAINSET
        else "non-official validation-derived train/val fallback"
    ),
    "paper_dataset": "https://github.com/PeterWang512/CNNDetection/tree/master/dataset",
    "download_full_trainset": DOWNLOAD_FULL_TRAINSET,
    "download_full_testset": DOWNLOAD_FULL_TESTSET,
    "caps": {
        "max_train_per_category": MAX_TRAIN_PER_CATEGORY,
        "max_val_per_category": MAX_VAL_PER_CATEGORY,
        "max_test_per_leaf": MAX_TEST_PER_LEAF,
        "n_ffhq_real": N_FFHQ_REAL,
    },
    "counts": counts,
    "data_dir": str(DATA_DIR),
}
(DATA_DIR / "cnndetection_manifest.json").write_text(json.dumps(manifest, indent=2))
print("Saved", DATA_DIR / "cnndetection_manifest.json")


In [ ]:
# 11. Preview a few images
from PIL import Image
import matplotlib.pyplot as plt

samples = []
for split in ["train", "val", "test"]:
    for label in ["real", "fake"]:
        files_ = image_files(DATA_DIR / split / label)
        samples.extend([(split, label, p) for p in files_[:2]])

fig, axes = plt.subplots(2, 6, figsize=(18, 7))
for ax, (split, label, path) in zip(axes.ravel(), samples[:12]):
    ax.imshow(Image.open(path).convert("RGB"))
    ax.set_title(f"{split}/{label}")
    ax.axis("off")
plt.tight_layout()
plt.show()


# Done

Reals: **LSUN** (CNNDetection) + **FFHQ**.

Fakes: CNNDetection **ProGAN** (train/val) and the official **13-generator test set** (if enabled).

Next: run `notebooks/train_colab_RUN2_combined.ipynb` on a GPU runtime. Keep the manifest: it records whether the official three-way protocol was used.
